In [1]:
import pandas as pd
import re
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import json
from pathlib import Path
import numpy as np
from tqdm.auto import tqdm
import torch
import torch.nn.functional as F

/Users/mnatali/Projects/sentiment_analysis/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipping import of cpp extensions due to incompatible torch version 2.9.1 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
W0729 11:24:36.106000 77332 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
language_classifier = pipeline(
    "text-classification",
    model = "papluca/xlm-roberta-base-language-detection"
)

def lang_result(text):
    results = language_classifier(
        text,
        truncation=True
    )
    return results[0]["label"]

theme_classifier = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/deberta-v3-base-zeroshot-v2.0",
    multi_label = True
)

def theme_result(text, theme_labels):
    return theme_classifier(
        text,
        candidate_labels=theme_labels,
        hypothesis_template="This post discusses {}.",
        multi_label=True
    )

model_name = "yangheng/deberta-v3-base-absa-v1.1"

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
model.eval()


def aspect_sentiment(text, aspect, batch_size=16, max_length=512, stride=64):
    encoded = tokenizer(
        text,
        aspect,
        truncation=True,
        max_length=max_length,
        stride=stride,
        return_overflowing_tokens=True,
        padding=True,
        return_tensors="pt"
    )

    input_keys = ["input_ids", "attention_mask", "token_type_ids"]
    input_keys = [k for k in input_keys if k in encoded]

    all_probs = []

    with torch.inference_mode():
        n_chunks = encoded["input_ids"].shape[0]

        for start in range(0, n_chunks, batch_size):
            end = start + batch_size

            batch = {
                k: encoded[k][start:end].to(device)
                for k in input_keys
            }

            outputs = model(**batch)
            probs = F.softmax(outputs.logits, dim=-1)
            all_probs.append(probs)

    avg_probs = torch.cat(all_probs, dim=0).mean(dim=0).cpu()

    return {
        model.config.id2label[i]: float(avg_probs[i])
        for i in range(len(avg_probs))
    }

Device set to use mps:0
Device set to use mps:0


Using device: mps


/Users/mnatali/Projects/sentiment_analysis/.venv/lib/python3.13/site-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [ ]:
def remove_links(text):
    url_pattern = re.compile(r'[\[(]?(?:https?://|www\.)\S+[\])]?' )
    return url_pattern.sub('', text)

In [4]:
BASE_DIR = Path.cwd()
file_path = (
    BASE_DIR
    / "brightdata_social_exports"
    / "instagram_datacenters_posts.json"
)
with file_path.open("r", encoding="utf-8") as f:
    instagram_posts = json.load(f)

In [5]:
english_post_ids = []
a = 0

for ig_post in instagram_posts:
    unclean_text = ig_post["description"]
    text = remove_links(unclean_text)
    language = lang_result(text)
    pid = ig_post["post_id"]
    if language == 'en':
        english_post_ids.append(pid)
    a += 1
    print("Posts scanned:", a, end="\r")

KeyboardInterrupt: 

In [6]:
print(len(english_post_ids))

7


In [7]:
all_post_ids = []

env_post_ids = []
env_post_sentiments = []
env_post_sentiment_degrees = []

infr_post_ids = []
infr_post_sentiments = []
infr_post_sentiment_degrees = []

housing_post_ids = []
housing_post_sentiments = []
housing_post_sentiment_degrees = []

econ_post_ids = []
econ_post_sentiments = []
econ_post_sentiment_degrees = []

life_qual_post_ids = []
life_qual_post_sentiments = []
life_qual_post_sentiment_degrees = []

aesth_post_ids = []
aesth_post_sentiments = []
aesth_post_sentiment_degrees = []

gov_post_ids = []
gov_post_sentiments = []
gov_post_sentiment_degrees = []

tech_post_ids = []
tech_post_sentiments = []
tech_post_sentiment_degrees = []

not_useful_post_ids = []

themes = ["visual impact of datacenters", "infrastructure and house utilities", "housing costs and property values", "economy and jobs", "quality of life, noise, and light pollution", "environmental impact", "government decisions and policies", "technology performance and growth"]

matched_posts = 0
more_than_one_theme_posts = 0

a = 0

for ig_post in instagram_posts:

    post_id = ig_post["post_id"]
    if post_id not in english_post_ids:
        continue
    all_post_ids.append(post_id)
    unclean_text = ig_post["description"]
    text = remove_links(unclean_text)


    final_labels = []

    theme_scores = theme_result(
        text,
        themes
    )
    
    for i in range(len(theme_scores['labels'])):
        if theme_scores['scores'][i] > 0.5:
            final_labels.append(theme_scores['labels'][i])
    
    if len(final_labels) > 0:
        matched_posts += 1
    else:
        not_useful_post_ids.append(post_id)
    
    if len(final_labels) > 1:
        more_than_one_theme_posts += 1
    

    for label in final_labels:
        total_sentiment = aspect_sentiment(text, label)
        post_sentiment = max(total_sentiment, key=total_sentiment.get)
        post_degree = max(total_sentiment.values())

        if label == "visual impact of datacenters":
            aesth_post_ids.append(post_id)
            aesth_post_sentiments.append(post_sentiment)
            aesth_post_sentiment_degrees.append(post_degree)
        if label == "infrastructure and house utilities":
            infr_post_ids.append(post_id)
            infr_post_sentiments.append(post_sentiment)
            infr_post_sentiment_degrees.append(post_degree)
        if label == "housing costs and property values":
            housing_post_ids.append(post_id)
            housing_post_sentiments.append(post_sentiment)
            housing_post_sentiment_degrees.append(post_degree)
        if label == "economy and jobs":
            econ_post_ids.append(post_id)
            econ_post_sentiments.append(post_sentiment)
            econ_post_sentiment_degrees.append(post_degree)
        if label == "quality of life, noise, and light pollution":
            life_qual_post_ids.append(post_id)
            life_qual_post_sentiments.append(post_sentiment)
            life_qual_post_sentiment_degrees.append(post_degree)
        if label == "environmental impact":
            env_post_ids.append(post_id)
            env_post_sentiments.append(post_sentiment)
            env_post_sentiment_degrees.append(post_degree)
        if label == "government decisions and policies":
            gov_post_ids.append(post_id)
            gov_post_sentiments.append(post_sentiment)
            gov_post_sentiment_degrees.append(post_degree)
        if label == "technology performance and growth":
            tech_post_ids.append(post_id)
            tech_post_sentiments.append(post_sentiment)
            tech_post_sentiment_degrees.append(post_degree)

        a += 1
        print("Posts scanned:", a, end="\r")

print("Total posts scanned:", len(all_post_ids))
print("Total posts with a theme:", matched_posts)
print("Found environmental posts:", len(env_post_ids))
print("Found infrastructure posts:", len(infr_post_ids))
print("Found housing posts:", len(housing_post_ids))
print("Found economic posts:", len(econ_post_ids))
print("Found life quality posts:", len(life_qual_post_ids))
print("Found aesthetic posts:", len(aesth_post_ids))
print("Found government posts:", len(gov_post_ids))
print("Found technological posts:", len(tech_post_ids))
print(not_useful_post_ids)

print(gov_post_sentiments)
print(gov_post_sentiment_degrees)

Total posts scanned: 7
Total posts with a theme: 5
Found environmental posts: 0
Found infrastructure posts: 0
Found housing posts: 0
Found economic posts: 0
Found life quality posts: 0
Found aesthetic posts: 0
Found government posts: 0
Found technological posts: 5
['3721327515874322375', '3662771468539466225']
[]
[]


200 posts scanned -> 123 in english
    -> When technology wasn't a theme and "infrastructure and utilities" was, 58 posts had a theme and 44 of them were infrastructure
    -> When technology was a theme and so was "infrastructure and utilities," 100 posts had a theme, 44 of them were infrastructure, and 77 were technology
    -> When technology was a theme and "infrastructure and house utilities" was, 93 posts had a theme, 3 of them were infrastructure, and 77 were technology

In [8]:
posts_by_id = {post["post_id"]: post for post in instagram_posts}
tech_links = []

for id in tech_post_ids:
    post = posts_by_id.get(id)
    tech_links.append(post["url"])

print(tech_links)

['https://www.instagram.com/p/DT4PU6AFFfU', 'https://instagram.com/p/DO_Znl6AWTl', 'https://www.instagram.com/p/DBNSoA-thC7', 'https://www.instagram.com/p/DDG2oDFNgKc', 'https://www.instagram.com/p/DTR3U4ujkvb/']


In [9]:
theme_lists = [env_post_ids, infr_post_ids, housing_post_ids, econ_post_ids, life_qual_post_ids, aesth_post_ids, gov_post_ids, tech_post_ids]
posts = pd.DataFrame(columns=["ids", "text", "date", "likes", "number of comments", "has photos", "has videos", "location", "followers", "is paid partnership", "environment", "infrastructure", "housing", "economy", "life quality", "aesthetics", "government", "technology", "AWS", "Amazon", "Google", "Microsoft", "Azure", "Meta", "Oracle", "Equinix", "Digital Realty", "IBM", "Facebook", "Apple", "QTS", "Vantage", "CyrusOne", "CoreSite"])
datacenters_keywords = ["datacenter", "data center", "datacentre", "data centre"]

posts_by_id = {post["post_id"]: post for post in instagram_posts}

for theme in theme_lists:
    df1 = pd.DataFrame(columns=["ids", "text", "date", "likes", "number of comments", "has photos", "has videos", "location", "followers", "is paid partnership", "environment", "infrastructure", "housing", "economy", "life quality", "aesthetics", "government", "technology", "AWS", "Amazon", "Google", "Microsoft", "Azure", "Meta", "Oracle", "Equinix", "Digital Realty", "IBM", "Facebook", "Apple", "QTS", "Vantage", "CyrusOne", "CoreSite"])
    post_ids = []
    post_texts = []
    post_dates = []
    post_likes = []
    post_comment_numbers = []
    post_has_photos = []
    post_has_videos = []
    post_locations = []
    post_user_followers = []
    post_is_paid_partnership = []


    for pid in theme:
        post_ids.append(pid)
        post = posts_by_id.get(pid)
        post_texts.append(remove_links(post["description"]))
        post_dates.append(post["date_posted"])

        if(post["likes"] == None):
            post_likes.append(0)
        else:
            post_likes.append(post["likes"])
        
        post_comment_numbers.append(post["num_comments"])

        if(post["photos"] == None):
            post_has_photos.append(False)
        else:
            post_has_photos.append(True)
        
        if(post["videos"] == None):
            post_has_videos.append(False)
        else:
            post_has_videos.append(True)
        
        post_locations.append(post["location"])

        if(post["followers"] == None):
            post_user_followers.append(0)
        else:
            post_user_followers.append(post["followers"])
        
        post_is_paid_partnership.append(post["is_paid_partnership"])

    df1["ids"] = post_ids
    df1["text"] = post_texts
    df1["date"] = post_dates
    df1["likes"] = post_likes
    df1["number of comments"] = post_comment_numbers
    df1["has photos"] = post_has_photos
    df1["has videos"] = post_has_videos
    df1["location"] = post_locations
    df1["followers"] = post_user_followers
    df1["is paid partnership"] = post_is_paid_partnership

    
    for col in ["environment", "infrastructure", "housing", "economy", "life quality", "aesthetics", "government", "technology"]:
        df1[col] = False

    for col in ["environment sentiment", "environment sentiment degree", "infrastructure sentiment", "infrastructure sentiment degree", "housing sentiment", "housing sentiment degree", "economy sentiment", "economy sentiment degree", "life quality sentiment", "life quality sentiment degree", "aesthetics sentiment", "aesthetics sentiment degree", "government sentiment", "government sentiment degree", "technology sentiment", "technology sentiment degree"]:
        df1[col] = None

    if theme == env_post_ids:
        df1["environment"] = True
        df1["environment sentiment"] = env_post_sentiments
        df1["environment sentiment degree"] = env_post_sentiment_degrees
    if theme == infr_post_ids:
        df1["infrastructure"] = True
        df1["infrastructure sentiment"] = infr_post_sentiments
        df1["infrastructure sentiment degree"] = infr_post_sentiment_degrees
    if theme == housing_post_ids:
        df1["housing"] = True
        df1["housing sentiment"] = housing_post_sentiments
        df1["housing sentiment degree"] = housing_post_sentiment_degrees
    if theme == econ_post_ids:
        df1["economy"] = True
        df1["economy sentiment"] = econ_post_sentiments
        df1["economy sentiment degree"] = econ_post_sentiment_degrees
    if theme == life_qual_post_ids:
        df1["life quality"] = True
        df1["life quality sentiment"] = life_qual_post_sentiments
        df1["life quality sentiment degree"] = life_qual_post_sentiment_degrees
    if theme == aesth_post_ids:
        df1["aesthetics"] = True
        df1["aesthetics sentiment"] = aesth_post_sentiments
        df1["aesthetics sentiment degree"] = aesth_post_sentiment_degrees
    if theme == gov_post_ids:
        df1["government"] = True
        df1["government sentiment"] = gov_post_sentiments
        df1["government sentiment degree"] = gov_post_sentiment_degrees
    if theme == tech_post_ids:
        df1["technology"] = True
        df1["technology sentiment"] = tech_post_sentiments
        df1["technology sentiment degree"] = tech_post_sentiment_degrees
    posts = pd.concat([posts, df1], ignore_index=True)

/var/folders/9m/h28gbbc970j03ncf7v7dhqk80000gq/T/ipykernel_77332/3579425662.py:103: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  posts = pd.concat([posts, df1], ignore_index=True)


In [10]:
len(posts)

5

In [11]:
posts = posts.astype({
    "ids": "string",
    "text": "string",
    "date": "string",
    "likes": "int64",
    "number of comments": "int64",
    "has photos": "bool",
    "has videos": "bool",
    "location": "string",
    "followers": "int64",
    "is paid partnership": "bool",
})

In [12]:
grouping_cols = ["ids", "text", "date", "likes", "number of comments", "has photos", "has videos", "location", "followers", "is paid partnership"]

theme_cols = ["environment", "infrastructure", "housing", "economy", "life quality", "aesthetics", "government", "technology"]
sent_cols  = ["environment sentiment", "environment sentiment degree", "infrastructure sentiment", "infrastructure sentiment degree", "housing sentiment", "housing sentiment degree", "economy sentiment", "economy sentiment degree", "life quality sentiment", "life quality sentiment degree", "aesthetics sentiment", "aesthetics sentiment degree", "government sentiment", "government sentiment degree", "technology sentiment", "technology sentiment degree"]

def first_non_null(s):
    return s.dropna().iloc[0] if s.notna().any() else np.nan

agg = {c: "max" for c in theme_cols}              # True if any True
agg.update({c: first_non_null for c in sent_cols}) # keep the real sentiment if present

posts = posts.groupby(grouping_cols, as_index=False, dropna=False).agg(agg)

In [13]:
len(posts)

8

In [14]:
posts.to_json('instagram_ABSA_entire_dataframe.json', orient='records', indent=4)

In [15]:
posts.head(30)

,ids,text,date,likes,number of comments,has photos,has videos,location,followers,is paid partnership,...,economy sentiment,economy sentiment degree,life quality sentiment,life quality sentiment degree,aesthetics sentiment,aesthetics sentiment degree,government sentiment,government sentiment degree,technology sentiment,technology sentiment degree
0,3480520001695649979,"NEXTDC pays $353m for Western Sydney site, lay...",2024-10-17T01:58:59.000Z,8,0,True,False,<NA>,18118,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Neutral,0.507566
1,3514736805808571036,New AI-designed carbon-removal material to be ...,2024-12-03T07:01:41.000Z,0,0,True,False,<NA>,10746,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Neutral,0.796791
2,3658037065625228868,"Through its unit Edgnex Data Centers, Damac wi...",2025-06-19T00:13:41.000Z,133,0,True,False,<NA>,82686,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Neutral,0.723729
3,3728811688410047717,✅ Mission accomplished!\n\nThe LORCO Data team...,2025-09-24T15:50:13.000Z,4,0,True,False,"['Orlando', 'Florida']",20,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Positive,0.895049
4,3763915258096812122,A major investment for India's digital future!...,2025-11-12T02:14:45.000Z,0,6,True,False,<NA>,592550,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Positive,0.903808
5,3778159852368000408,Want to know where I am looking to invest next...,2025-12-01T17:56:13.000Z,54,8,True,False,<NA>,112186,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Positive,0.925009
6,3806066487963044827,Now is the time to upgrade your hosting experi...,2026-01-09T06:01:49.000Z,1,0,True,False,<NA>,1374,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Positive,0.923337
7,3816868091562055636,1) Infrastructure and efficiency gains: Google...,2026-01-24T03:42:44.000Z,0,0,True,False,<NA>,22,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Positive,0.908669


In [16]:
print(posts.columns)

Index(['ids', 'text', 'date', 'likes', 'number of comments', 'has photos',
       'has videos', 'location', 'followers', 'is paid partnership',
       'environment', 'infrastructure', 'housing', 'economy', 'life quality',
       'aesthetics', 'government', 'technology', 'environment sentiment',
       'environment sentiment degree', 'infrastructure sentiment',
       'infrastructure sentiment degree', 'housing sentiment',
       'housing sentiment degree', 'economy sentiment',
       'economy sentiment degree', 'life quality sentiment',
       'life quality sentiment degree', 'aesthetics sentiment',
       'aesthetics sentiment degree', 'government sentiment',
       'government sentiment degree', 'technology sentiment',
       'technology sentiment degree'],
      dtype='object')


In [17]:
# calculating average sentiment based on theme:
# Weigh all posts by their degree in the numerator and denominator, means that the average sentiment will just be +/- 1 if there are only positive or negative themes but other than that does a pretty good job of weighing neutrality
def avg_sentiment_calculation(theme):
    theme_posts = posts[posts[theme] == True]
    if len(theme_posts) > 0:
        pos = theme_posts.loc[theme_posts[f"{theme} sentiment"] == "Positive", f"{theme} sentiment degree"].sum()
        neg = theme_posts.loc[theme_posts[f"{theme} sentiment"] == "Negative", f"{theme} sentiment degree"].sum()
        total = theme_posts[f"{theme} sentiment degree"].sum()
        return len(theme_posts), (pos-neg)/total
    else:
        return 0, None


print("Number of environmental posts: ", avg_sentiment_calculation("environment")[0], ", Average sentiment of environmental posts: ", avg_sentiment_calculation("environment")[1], sep="")
print("Number of infrastructural posts: ", avg_sentiment_calculation("infrastructure")[0], ", Average sentiment of infrastructural posts: ", avg_sentiment_calculation("infrastructure")[1], sep="")
print("Number of housing-related posts: ", avg_sentiment_calculation("housing")[0], ", Average sentiment of housing-related posts: ", avg_sentiment_calculation("housing")[1], sep="")
print("Number of economic posts: ", avg_sentiment_calculation("economy")[0], ", Average sentiment of economic posts: ", avg_sentiment_calculation("economy")[1], sep="")
print("Number of life-quality-related posts: ", avg_sentiment_calculation("life quality")[0], ", Average sentiment of life-quality-related posts: ", avg_sentiment_calculation("life quality")[1], sep="")
print("Number of aesthetics-related posts: ", avg_sentiment_calculation("aesthetics")[0], ", Average sentiment of aesthetics-related posts: ", avg_sentiment_calculation("aesthetics")[1], sep="")
print("Number of governmental posts: ", avg_sentiment_calculation("government")[0], ", Average sentiment of governmental posts: ", avg_sentiment_calculation("government")[1], sep="")
print("Number of technological posts: ", avg_sentiment_calculation("technology")[0], ", Average sentiment of technological posts: ", avg_sentiment_calculation("technology")[1], sep="")


Number of environmental posts: 0, Average sentiment of environmental posts: None
Number of infrastructural posts: 0, Average sentiment of infrastructural posts: None
Number of housing-related posts: 0, Average sentiment of housing-related posts: None
Number of economic posts: 0, Average sentiment of economic posts: None
Number of life-quality-related posts: 0, Average sentiment of life-quality-related posts: None
Number of aesthetics-related posts: 0, Average sentiment of aesthetics-related posts: None
Number of governmental posts: 0, Average sentiment of governmental posts: None
Number of technological posts: 8, Average sentiment of technological posts: 0.6919654584835809


In [19]:
posts['year'] = pd.to_datetime(posts['date']).dt.year

year_datasets = {year: posts[posts['year'] == year] for year in range(2010, 2027)}

posts_2010 = year_datasets[2010]
posts_2020 = year_datasets[2020]